In [9]:
%pip install "llama-index-llms-openai==0.1.26"

Defaulting to user installation because normal site-packages is not writeable


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.8 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: llama-index-core
    Found existing installation: llama-index-core 0.14.26
    Uninstalling llama-index-core-0.14.26:
      Successfully uninstalled llama-index-core-0.14.26
  Attempting uninstall: llama-index-llms-openai
    Found existing installation: llama-index-llms-openai 0.9.0
    Uninstalling llama-index-llms-openai-0.9.0:
      Successfully uninstalled llama-index-llms-openai-0.9.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-embeddings-openai 0.8.0 requires llama-index-core<0.15,>=0.13.0, but you have llama-index-core 0.10.68.post1 which is incompatible.
llama-index 0.10.27 requires llama-index-embeddings-openai<0.2.0,>=0.1.5, but you have llama-index-embeddings-openai 0.8.0 which is incompatible.
llama-index-cli 0.1.12 requires

In [ ]:
# 1. Patch: register gpt-4o-mini with the older package (must run first)
from llama_index.llms.openai import utils as openai_utils

openai_utils.ALL_AVAILABLE_MODELS["gpt-4o-mini"] = 128000
openai_utils.CHAT_MODELS["gpt-4o-mini"] = 128000

# 2. Setup
from pathlib import Path

import nest_asyncio
from helper import get_openai_api_key

OPENAI_API_KEY = get_openai_api_key()
nest_asyncio.apply()

# 3. Papers (these files must already be in the working folder)
papers = [
    "12_Optimizing_the_Landscape_of.pdf",
    "34_The_Use_of_AI_Tools_to_Deve.pdf",
    "18_Automated_Item_Generation_f.pdf",
]

missing = [p for p in papers if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f"Missing PDF files: {missing}")

# 4. Build the tools
from utils import get_doc_tools

paper_to_tools_dict = {}
for paper in papers:
    print(f"Getting tools for paper: {paper}")
    vector_tool, summary_tool = get_doc_tools(paper, Path(paper).stem)
    paper_to_tools_dict[paper] = [vector_tool, summary_tool]

initial_tools = [t for paper in papers for t in paper_to_tools_dict[paper]]
print(f"Number of tools: {len(initial_tools)}")  # expected: 6

# 5. Build the agent
from llama_index.llms.openai import OpenAI
from llama_index.core.agent import FunctionCallingAgentWorker, AgentRunner

llm = OpenAI(model="gpt-4o-mini")

agent_worker = FunctionCallingAgentWorker.from_tools(
    initial_tools,
    llm=llm,
    verbose=True,
)
agent = AgentRunner(agent_worker)

# 6. Ask questions
response = agent.query(
    "Tell me about the evaluation dataset used in Optimizing_the_Landscape_of, "
    "and then tell me about the evaluation results"
)
print(str(response))

response = agent.query(
    "Give me a summary of both Automated_Item_Generation_f "
    "and The_Use_of_AI_Tools_to_Deve"
)
print(str(response))

Getting tools for paper: 12_Optimizing_the_Landscape_of.pdf
Getting tools for paper: 34_The_Use_of_AI_Tools_to_Deve.pdf
Getting tools for paper: 18_Automated_Item_Generation_f.pdf
Number of tools: 6
Added user message to memory: Tell me about the evaluation dataset used in Optimizing_the_Landscape_of, and then tell me about the evaluation results
=== Calling Function ===
Calling function: vector_tool_12_Optimizing_the_Landscape_of with args: {"query": "evaluation dataset"}
=== Function Output ===
The provided information does not specify an evaluation dataset. It focuses on the methodology and results of embedding models and simulations, including metrics like Normalized Mutual Information (NMI) and Total Entropy Fit Index (TEFI), but does not mention any specific dataset used for evaluation.
=== Calling Function ===
Calling function: vector_tool_12_Optimizing_the_Landscape_of with args: {"query": "evaluation results"}
=== Function Output ===
The evaluation results indicate that optimi